In [ ]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.datasets import imdb
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense

In [ ]:
vocab_size=10000
#Keep only the top 10000 most frequent words
(X_train, y_train), (X_test, y_test) = imdb.load_data(num_words=vocab_size)
print(' training sequences:', len(X_train))
print('test sequences:', len(X_test))

17464789/17464789 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
 training sequences: 25000
test sequences: 25000


In [ ]:
max_length=200
X_train=pad_sequences(X_train, maxlen=max_length,padding='post')
X_test=pad_sequences(X_test, maxlen=max_length,padding='post')

In [ ]:
model=Sequential([
    Embedding(
        input_dim=vocab_size,
        output_dim=64,
        input_length=max_length),
    SimpleRNN(64),
    Dense(32, activation='relu'),
    Dense(1, activation='sigmoid')
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:100: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [ ]:
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

In [ ]:
model.summary()

Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ simple_rnn (SimpleRNN)          │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

In [ ]:
history=model.fit(X_train,y_train,epochs=5,validation_split=0.2,batch_size=64)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 87ms/step - accuracy: 0.5103 - loss: 0.6941 - val_accuracy: 0.5144 - val_loss: 0.6891
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 25s 79ms/step - accuracy: 0.5683 - loss: 0.6728 - val_accuracy: 0.6268 - val_loss: 0.6371
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 44s 88ms/step - accuracy: 0.6029 - loss: 0.6381 - val_accuracy: 0.5830 - val_loss: 0.6435
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 27s 86ms/step - accuracy: 0.6425 - loss: 0.5780 - val_accuracy: 0.5960 - val_loss: 0.6572
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 28s 90ms/step - accuracy: 0.6569 - loss: 0.5468 - val_accuracy: 0.6268 - val_loss: 0.8445


In [ ]:
loss,accuracy=model.evaluate(X_test,y_test)
print('Test accuracy:', accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 12s 16ms/step - accuracy: 0.6218 - loss: 0.8445
Test accuracy: 0.6218000054359436


In [ ]:
from tensorflow.keras.datasets import imdb
word_index=imdb.get_word_index()

#shift indices by 3 because keras reserves 0,1,2
word_index={k:(v+3) for k,v in word_index.items()}
word_index['<PAD>']=0
word_index['<START>']=1
word_index['<UNK>']=2
word_index['<UNUSED>']=3

1641221/1641221 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [ ]:
def review_to_sequence(review):
  review=review.lower().split()
  sequence=[]
  for word in review:
    sequence.append(word_index.get(word,2)) #2=unknown word
  return sequence

In [ ]:
from tensorflow.keras.preprocessing.sequence import pad_sequences
import numpy as np
max_length=200
def predict_sentiment(review):
  sequence=review_to_sequence(review)
  padded=pad_sequences(
      [sequence],
      maxlen=max_length,
      padding='post',
      truncating='post'
  )
  prediction=model.predict(padded,verbose=0)
  score=prediction[0][0]
  print("Sentiment Score",score)
  if score>0.5:
    print("Positive review")
  else:
    print("Negative review")

In [ ]:
predict_sentiment("This movie is boring")

Sentiment Score 0.877976
Positive review
